# Member 8 (Week 3): Constraints and Normalisation with a Library Example

Your entire chunk in one sentence:

> **Normalisation decides WHERE each fact should live (one home only). Constraints are RULES that stop bad data from entering.**

Library version:

> **Each fact about a book, member or loan is stored once, in one place, and the database refuses to save anything that breaks the rules.**


## 1. Start from absolute zero: what is a table?

Forget SQL for 60 seconds. Think of a paper register at a library desk in Port Harcourt.

| Column = a question | Row = one answer set |
|---|---|
| What is the book title? | Clean Code |
| Who borrowed it? | Ada |
| When is it due? | 2026-11-20 |

In a database:

- **Table** = the whole register (e.g. `loans`).
- **Column** = one type of fact (e.g. `due_date`).
- **Row** = one real thing (e.g. one loan).
- **Cell** = one fact about one thing.

```text
Table: loans
+---------+----------+------------+
| book    | member   | due_date   |
+---------+----------+------------+
| Clean Code | Ada   | 2026-11-20 |  <- one ROW
| Half of a Yellow Sun | Chinedu | 2026-11-22 |  <- another ROW
+---------+----------+------------+
   ^         ^            ^
   columns
```

A database is just many such registers that **link to each other**. Normalisation tells us how many registers to have. Constraints are the strict librarian who checks every new line before it is written.

## 2. Meet our MESSY library spreadsheet (before normalisation)

Imagine the library starts with ONE giant table. This is what beginners always do. All names below are fictional.

```text
Table: messy_loans  (THIS IS BAD - on purpose)
+---------+--------------------------------+----------------------+---------------------------+------------+
| loan_id | book_info                      | member_info          | member_phone              | due_date   |
+---------+--------------------------------+----------------------+---------------------------+------------+
| 1       | Clean Code, Robert Martin      | Ada, GRA Phase 2     | 0803-000-0001, 0805-111-1111 | 2026-11-20 |
| 2       | Clean Code, Robert Martin      | Chinedu, D-line      | 0803-000-0002             | 2026-11-22 |
| 3       | Purple Hibiscus, Adichie       | Ada, GRA Phase 2     | 0803-000-0001, 0805-111-1111 | 2026-11-25 |
+---------+--------------------------------+----------------------+---------------------------+------------+
```

Look closely. Three terrible smells:

1. **One cell holds many facts:** `Clean Code, Robert Martin` - title AND author jammed together. Two phone numbers in one cell.
2. **Same fact repeated:** `Clean Code, Robert Martin` appears twice. Ada's address and phones appear twice.
3. **No rules:** nothing stops someone from typing `due_date = 1800-01-01` or borrowing book ID `99999` that does not exist.

This table is in what teachers call **0NF (unnormalised)**. It works for 10 rows. It becomes hell at 10,000 rows.

Keep this ugly table in your head. The rest of this lesson fixes it step by step.

## 3. Why duplicated data causes bugs: the 3 anomaly stories

This is the heart of your chunk. Your README says: *why duplicated data causes bugs*. There are exactly 3 bugs. Learn them as stories.

### Bug 1: Update anomaly (you update in one place, forget the other)

Ada moves from GRA Phase 2 to Peter Odili Road. The librarian updates loan row 1 but forgets row 3.

```text
| 1 | ... | Ada, Peter Odili Road | ... |
| 3 | ... | Ada, GRA Phase 2      | ... |  <- STALE! Which address is true?
```

Now the database **contradicts itself**. Fine notices go to the wrong address. This is an **update anomaly**.

Root cause: Ada's address lives in TWO places. One truth, two copies = they can disagree.

### Bug 2: Insert anomaly (you cannot add a fact without a fake row)

The library buys a new book: `The Famished Road`. Nobody has borrowed it yet. Where do you store its title and author?

Our only table is `messy_loans`. Every row needs a loan_id, member and due date. So you must invent a **fake loan** just to store a book. Silly and dangerous.

Root cause: books and loans are jammed in one table. You cannot record one without the other.

### Bug 3: Delete anomaly (deleting one fact accidentally deletes another)

Chinedu returns his book and his loan row (row 2) is deleted. But wait - if that was the ONLY row mentioning his phone number, his contact is now **gone forever**. Deleting a loan deleted a member.

Root cause: two different facts (loan happened + member's phone) share one row. One row, two lives.

> **Memory trick for class: U-I-D. Update, Insert, Delete. Duplication rots Updates, blocks Inserts, and murders Deletes.**

Normalisation exists to kill all three.

## 4. The fix in one idea: one fact, one home

The cure for all 3 bugs is a single rule:

> **Every fact lives in ONE place only. Everywhere else just POINTS to it.**

In library terms:

- Ada's address lives in the `members` table ONLY. Loan rows just say `member_id = 7`.
- A book's title/author live in the `books` table ONLY. Loan rows just say `book_id = 42`.
- A loan's due date lives in the `loans` table ONLY.

Two tools enforce this:

| Tool | Job | Analogy |
|---|---|---|
| **Normalisation** (1NF, 2NF, 3NF) | Decides how to SPLIT the giant table into small clean tables | Deciding how many registers the library needs |
| **Constraints** (keys, checks) | RULES that reject bad writes | The strict head librarian who says NO |

Next: the 3 normal forms, each fixing one type of mess. They build on each other: you must pass 1NF before 2NF matters, and 2NF before 3NF matters.

## 5. 1NF - First Normal Form: one cell, one fact

**Rule: every cell holds ONE single (atomic) value. No lists, no comma-jammed facts. Every row is uniquely identifiable.**

### Violation (from our messy table)

```text
| book_info                  | member_phone                 |
|----------------------------|------------------------------|
| Clean Code, Robert Martin  | 0803-000-0001, 0805-111-1111 |
```

Why is this evil? Try: *find all loans for books by Robert Martin*. The database must parse text with commas. Try: *text Ada on her second number*. Which half of the cell is it? Pain.

### Fix: split columns, split rows

```text
Table: loans_1nf (better, not perfect yet)
+---------+------------+----------------+-----------+----------------+---------------+------------+
| loan_id | book_title | book_author    | member_name | member_address | member_phone  | due_date   |
+---------+------------+----------------+-----------+----------------+---------------+------------+
| 1       | Clean Code | Robert Martin  | Ada       | GRA Phase 2    | 0803-000-0001 | 2026-11-20 |
| 1b      | Clean Code | Robert Martin  | Ada       | GRA Phase 2    | 0805-111-1111 | 2026-11-20 |
| 2       | Clean Code | Robert Martin  | Chinedu   | D-line         | 0803-000-0002 | 2026-11-22 |
+---------+------------+----------------+-----------+----------------+---------------+------------+
```

Now each cell is atomic. One phone per row (or better: a separate `member_phones` table - same idea).

Plus: `loan_id` uniquely identifies each row. That is the start of a PRIMARY KEY (more in section 10).

> **1NF in one line: no lists in cells; every row has an ID.**

But notice: `Clean Code / Robert Martin` STILL repeats. `Ada / GRA Phase 2` STILL repeats. 1NF fixed the cells, not the duplication. That needs 2NF and 3NF.

## 6. 2NF - Second Normal Form: the whole key, nothing but the key

**Rule: must be in 1NF, AND every non-key column must depend on the WHOLE primary key, not just part of it.**

When does this bite? When the primary key has **two parts** (a composite key). Library example:

Suppose someone designs:

```text
Table: loan_items (composite key: loan_id + book_id)
+---------+---------+------------+---------------+------------+
| loan_id | book_id | book_title | book_author   | due_date   |
+---------+---------+------------+---------------+------------+
| 1       | 42      | Clean Code | Robert Martin | 2026-11-20 |
| 1       | 43      | Purple Hibiscus | Adichie    | 2026-11-20 |
+---------+---------+------------+---------------+------------+
PRIMARY KEY (loan_id, book_id)
```

Ask: what does `book_title` depend on? Just `book_id`! Book 42 is always Clean Code, in every loan. It does NOT need `loan_id`. That is a **partial dependency** - depending on PART of the key.

Consequence: `Clean Code / Robert Martin` repeats in every loan of that book. Update anomaly returns: fix a typo in the title and you must hunt every loan row.

### Fix: move the partially-dependent facts to their own table

```text
Table: books              Table: loan_items
+---------+------------+    +---------+---------+------------+
| book_id | title      |    | loan_id | book_id | due_date   |
| (PK)    |            |    | (PKs)   | (FK)    |            |
+---------+------------+    +---------+---------+------------+
| 42      | Clean Code |    | 1       | 42      | 2026-11-20 |
| 43      | Purple Hibiscus | | 1       | 43      | 2026-11-20 |
+---------+------------+    +---------+---------+------------+
```

Title now lives ONCE in `books`. Loan rows just point with `book_id`.

> **2NF in one line: if a column only needs PART of the key, move it to the table for that part.**

Shortcut most professionals use: give every table a **single-column surrogate key** (e.g. `id SERIAL PRIMARY KEY`). Then partial dependencies become impossible and 2NF is automatic. Your library schema will do exactly this.

## 7. 3NF - Third Normal Form: no gossip chains

**Rule: must be in 2NF, AND no non-key column depends on ANOTHER non-key column (no transitive dependency).**

Fancy words, simple idea. Bad design:

```text
Table: members (violates 3NF)
+-----------+-----------+----------------+---------------+------------------+
| member_id | name      | address        | city          | city_postcode    |
+-----------+-----------+----------------+---------------+------------------+
| 7         | Ada       | GRA Phase 2    | Port Harcourt | 500001           |
| 8         | Chinedu   | D-line         | Port Harcourt | 500001           |
+-----------+-----------+----------------+---------------+------------------+
```

Chain of gossip: `member_id -> city -> city_postcode`. The postcode depends on the CITY, not directly on the member. Every Port Harcourt member repeats `500001`. If the postcode changes, you update 5,000 rows. Update anomaly again.

### Fix: move the gossip to its own table

```text
Table: members                    Table: cities
+-----------+-----------+--------+    +---------------+----------+
| member_id | name      | city   |    | city (PK)     | postcode |
+-----------+-----------+--------+    +---------------+----------+
| 7         | Ada       | PH     |    | PH            | 500001   |
| 8         | Chinedu   | PH     |    | Lagos         | 100001   |
+-----------+-----------+--------+    +---------------+----------+
```

Now the postcode lives ONCE per city.

> **3NF in one line: non-key facts must depend on THE KEY, the WHOLE key, and NOTHING BUT the key.** (Old teachers joke: like a court oath.)

### The famous summary of all three

| Form | Demands | Kills |
|---|---|---|
| 1NF | One value per cell, rows identifiable | Unsearchable comma-lists |
| 2NF | No partial-key dependency | Book facts repeated per loan |
| 3NF | No non-key depending on non-key | City facts repeated per member |

### Do you need 4NF, 5NF, BCNF?

For this course: NO. Know they exist (BCNF is a stricter 3NF; 4NF/5NF handle exotic multi-value cases). In real work, **3NF is the sweet spot**. If anyone asks in your question round, say: *we stop at 3NF because it removes the practical duplication bugs without over-fragmenting the schema.* That is a trade-off (your team rules require trade-offs in answers).

## 8. Our clean library schema (the destination)

After applying 1NF -> 2NF -> 3NF to the messy spreadsheet, we get the Month-1 library design. Five tables (Member 6 owns the full schema; you must be able to JUSTIFY it):

```text
  members 1----* loans *----1 books 1----* copies
                    |
                    1
                    |
                    *
                  fines
```

| Table | One row means | Key facts (each lives HERE only) |
|---|---|---|
| `books` | One title the library stocks | title, author, isbn (unique) |
| `copies` | One physical copy on a shelf | barcode (unique), condition, book_id -> books |
| `members` | One registered borrower | name, email (unique), joined_date |
| `loans` | One borrowing event | member_id -> members, copy_id -> copies, loan_date, due_date, return_date (NULL = still out) |
| `fines` | One fine for a late loan | loan_id -> loans, amount_cents, paid |

Why separate `books` and `copies`? Because the library can own 5 copies of Clean Code. Title/author belong to the BOOK (one fact). Condition/shelf belong to each COPY. Merging them would violate 2NF/3NF and resurrect duplication.

Why store money as `amount_cents INTEGER`? Because decimal money (0.1 + 0.2) breaks in floating point. Member 3's chunk covers this - but you should nod to it: integers are exact, floats are approximate. A fine of 250 naira = 25000 kobo stored as integer.

Notice the arrows (`->`). Those are FOREIGN KEYS - the constraints that glue clean tables together. Without them, split tables would drift apart. That is why YOUR chunk pairs normalisation WITH constraints.

## 9. Constraints: the strict head librarian

Normalisation splits tables nicely. But nice tables mean NOTHING if the app can still write garbage. Enter **constraints**.

> **A constraint is a rule the DATABASE enforces. It rejects any INSERT or UPDATE that breaks it - no matter how buggy the app is.**

Why enforce rules in the database instead of just in Python?

| Check in Python only | Check as DB constraint |
|---|---|---|
| Every app, script, admin tool must remember the rule | Rule lives in ONE place, enforced always |
| Two requests at the same instant can both pass the check, then both write (race condition) | The database locks correctly; the loser gets rejected |
| A bug or a raw SQL session bypasses it | NOTHING bypasses it (except the superuser disabling it) |

Professional rule: **validate in BOTH**. Python/Pydantic gives friendly errors fast; the database constraint is the final iron gate that never sleeps. Never trust the app alone for data integrity.

The 5 constraints you must know cold:

| Constraint | One-line meaning | Library example |
|---|---|---|
| `PRIMARY KEY` | Unique ID for each row, never NULL | `loans.loan_id` |
| `FOREIGN KEY` | This value MUST exist in another table | `loans.member_id` must exist in `members` |
| `UNIQUE` | No two rows share this value | `books.isbn`, `members.email` |
| `NOT NULL` | This column must always have a value | `loans.due_date` |
| `CHECK` | Custom yes/no rule | `fines.amount_cents >= 0` |

(Plus `DEFAULT`: a value filled in automatically, e.g. `loan_date DEFAULT CURRENT_DATE`.)

Next three sections go constraint by constraint with copy-paste Postgres SQL.

## 10. PRIMARY KEY, UNIQUE and NOT NULL

### PRIMARY KEY: the row's identity card

```sql
CREATE TABLE members (
  member_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  full_name TEXT NOT NULL,
  email TEXT NOT NULL UNIQUE
);
```

- `PRIMARY KEY` = UNIQUE + NOT NULL + an index, all in one. Each row gets exactly one identity.
- `GENERATED ALWAYS AS IDENTITY` is modern Postgres for auto-numbering (1, 2, 3...). Older tutorials say `SERIAL` - same idea, older syntax.
- Every table should have a primary key. No exceptions in this course. Why? Without one you cannot point at a row, update exactly one row, or be a foreign-key target.

**Natural vs surrogate key (classic interview question):**

- Natural key = a real fact as ID, e.g. ISBN for books, email for members.
- Surrogate key = a meaningless number (`book_id 42`) with the real fact kept UNIQUE.
- **We use surrogate keys** because real facts CHANGE (people change emails; books get re-issued ISBNs). Changing a primary key means updating every foreign key pointing at it - horrible. Numbers never need changing. Trade-off: one extra column and join, but total stability. Always worth it.

### UNIQUE: no duplicates allowed

- `email TEXT NOT NULL UNIQUE` - two members cannot share an email. The second INSERT is rejected.
- `isbn TEXT UNIQUE` - same ISBN cannot be entered twice (but NULL is allowed twice! In Postgres, NULLs are not considered equal, so multiple NULL ISBNs are fine - only real values must differ).
- Postgres builds an index behind UNIQUE, so lookups by email/ISBN are also fast. Two wins.

### NOT NULL: this fact is mandatory

- `due_date DATE NOT NULL` - a loan without a due date is nonsense, so forbid it.
- `return_date DATE NULL` (or just `DATE`) - NULL means *still on loan*. NULL is not zero or empty string; it means **unknown / not applicable**. Member 1's chunk (NULL behaviour) owns this deeply - you just need: allow NULL only where *missing* is a valid real state.

> **Teach it to a child: PRIMARY KEY = your fingerprint. UNIQUE = no two pupils share a desk. NOT NULL = you must write your name on the exam.**

## 11. FOREIGN KEY: the glue (and your Friday build)

The most important constraint in your chunk. It says:

> **This column's value MUST already exist as a key in another table.**

Library SQL:

```sql
CREATE TABLE loans (
  loan_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  member_id BIGINT NOT NULL REFERENCES members(member_id),
  copy_id BIGINT NOT NULL REFERENCES copies(copy_id),
  loan_date DATE NOT NULL DEFAULT CURRENT_DATE,
  due_date DATE NOT NULL,
  return_date DATE,  -- NULL = book still out
  CONSTRAINT due_after_loan CHECK (due_date > loan_date)
);
```

`member_id ... REFERENCES members(member_id)` is the foreign key. Vocabulary:

- **Child table** (`loans`) holds the foreign key. **Parent table** (`members`) holds the primary key.
- The database now REFUSES: a loan for member 9999 if member 9999 does not exist. It also refuses deleting member 7 while her loans still point at her (unless you say what to do - next section).

### Your Friday build, exactly

Your task: *Add a foreign key and show a rejected bad insert.* That means this demo:

```sql
-- 1. Add the foreign key to an existing table (ALTER TABLE form):
ALTER TABLE loans
  ADD CONSTRAINT fk_loans_member
  FOREIGN KEY (member_id) REFERENCES members(member_id);

-- 2. Good insert: member 7 exists -> accepted
INSERT INTO loans (member_id, copy_id, due_date) VALUES (7, 3, '2026-11-20');

-- 3. Bad insert: member 9999 does NOT exist -> REJECTED
INSERT INTO loans (member_id, copy_id, due_date) VALUES (9999, 3, '2026-11-20');
-- ERROR: insert or update on table "loans" violates foreign key constraint "fk_loans_member"
-- DETAIL: Key (member_id)=(9999) is not present in table "members".
```

That ERROR is the whole point. Screenshot it. It proves the database - not your Python - guards integrity. Your reviewer will ask: *what if the app checked first?* Answer: between the app's check and its insert, another request could delete the member (race). Only the database sees the true final order. Trade-off acknowledged: the DB error is uglier than a Pydantic message, so do BOTH - friendly app validation + iron DB constraint.

### Do foreign keys need an index?

Postgres does NOT auto-index foreign keys (unlike primary keys). If you often find *all loans of member 7* or delete members, add `CREATE INDEX ON loans(member_id);` or deletes/joins get slow. Member 4 (indexes/EXPLAIN) owns the deep story; you just need the one-liner.

## 12. What happens on DELETE? CASCADE vs RESTRICT vs SET NULL

A foreign key also controls deletes/updates of the PARENT. Three policies - you must pick the right one per relationship:

| Policy | Meaning | Library use |
|---|---|---|
| `ON DELETE RESTRICT` (the default) | Refuse to delete a parent that still has children | **Members, copies:** cannot delete Ada while her loans exist. History must survive. |
| `ON DELETE CASCADE` | Deleting the parent auto-deletes its children | **Rare in a library!** Maybe `loan_items` rows die with their loan. NEVER cascade members->loans (deleting a member must not erase borrowing history/fines). |
| `ON DELETE SET NULL` | Parent deleted; child's pointer becomes NULL | Only where NULL is meaningful, e.g. `copies.shelf_id` - demolish a shelf, copies become unshelved (NULL), not destroyed. |

SQL:

```sql
ALTER TABLE loans
  ADD CONSTRAINT fk_loans_member
  FOREIGN KEY (member_id) REFERENCES members(member_id)
  ON DELETE RESTRICT;  -- safe default: refuse
```

> **Exam answer: for loans we use RESTRICT because loans are financial/legal history. CASCADE would destroy evidence. The trade-off: you must return/archive loans before removing a member - slightly more work, but no silent data loss.**

Beginner trap: CASCADE feels convenient (*one delete cleans everything!*) until someone deletes one member and 3 years of fines vanish. In your question round, ask: *what would CASCADE on members do to the fines table?* Let the class feel the horror.

## 13. CHECK and DEFAULT: the fine print

### CHECK: any yes/no rule you can write

```sql
CREATE TABLE fines (
  fine_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  loan_id BIGINT NOT NULL REFERENCES loans(loan_id) ON DELETE RESTRICT,
  amount_cents INTEGER NOT NULL CHECK (amount_cents >= 0),
  paid BOOLEAN NOT NULL DEFAULT FALSE
);
```

More library CHECKs:

```sql
-- Due date must be after loan date (already seen)
CONSTRAINT due_after_loan CHECK (due_date > loan_date),
-- Return cannot precede the loan
CONSTRAINT return_after_loan CHECK (return_date IS NULL OR return_date >= loan_date),
-- Copy condition is a fixed menu
condition TEXT NOT NULL CHECK (condition IN ('new','good','worn','damaged')),
-- ISBN shape (13 digits) - keep the full validation in Pydantic too
CONSTRAINT isbn_shape CHECK (isbn ~ '^[0-9]{13}$')
```

Notes for teachers:

- CHECKs can span columns of the SAME row (`due_date > loan_date`) but cannot look at other rows or tables. Cross-row rules (e.g. *a member may hold at most 5 active loans*) need triggers or app logic + a careful transaction - say so honestly if asked.
- Name your constraints (`CONSTRAINT due_after_loan ...`) so error messages tell humans what broke. Unnamed constraints get ugly auto-names like `loans_check1`.

### DEFAULT: fill it in when the writer omits it

- `loan_date DATE NOT NULL DEFAULT CURRENT_DATE` - most loans start today; the app may omit it.
- `paid BOOLEAN NOT NULL DEFAULT FALSE` - new fines start unpaid.
- DEFAULT is convenience, not integrity. It does not validate - pair it with NOT NULL/CHECK.

> **Child version: CHECK = the rules pasted on the library wall. DEFAULT = the librarian filling in today's date for you when you forget.**

## 14. Full Postgres schema v1 (copy-paste starter)

Here is the whole clean design in one script. Member 6 will version this with Alembic; you need to READ it and justify every key. Fictional data only.

```sql
CREATE TABLE books (
  book_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  title TEXT NOT NULL,
  author TEXT NOT NULL,
  isbn TEXT UNIQUE CHECK (isbn ~ '^[0-9]{13}$')
);

CREATE TABLE copies (
  copy_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  book_id BIGINT NOT NULL REFERENCES books(book_id) ON DELETE RESTRICT,
  barcode TEXT NOT NULL UNIQUE,
  condition TEXT NOT NULL DEFAULT 'good'
    CHECK (condition IN ('new','good','worn','damaged'))
);

CREATE TABLE members (
  member_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  full_name TEXT NOT NULL,
  email TEXT NOT NULL UNIQUE,
  joined_date DATE NOT NULL DEFAULT CURRENT_DATE
);

CREATE TABLE loans (
  loan_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  member_id BIGINT NOT NULL REFERENCES members(member_id) ON DELETE RESTRICT,
  copy_id BIGINT NOT NULL REFERENCES copies(copy_id) ON DELETE RESTRICT,
  loan_date DATE NOT NULL DEFAULT CURRENT_DATE,
  due_date DATE NOT NULL,
  return_date DATE,
  CONSTRAINT due_after_loan CHECK (due_date > loan_date),
  CONSTRAINT return_after_loan CHECK (return_date IS NULL OR return_date >= loan_date)
);
CREATE INDEX ON loans(member_id);
CREATE INDEX ON loans(copy_id);

CREATE TABLE fines (
  fine_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  loan_id BIGINT NOT NULL REFERENCES loans(loan_id) ON DELETE RESTRICT,
  amount_cents INTEGER NOT NULL CHECK (amount_cents >= 0),
  paid BOOLEAN NOT NULL DEFAULT FALSE
);
```

Justification drill (say this out loud before class):

- *Why 5 tables?* 3NF: books vs copies vs members vs loans vs fines are different facts with different lives.
- *Why surrogate IDs?* Stability: names/emails/ISBNs can change; numbers never do.
- *Why RESTRICT everywhere?* Loans and fines are history/money: never silently delete them.
- *Why UNIQUE on email/barcode/isbn?* Real-world identity: no two members share an email, no two copies share a barcode.
- *Why money as INTEGER cents?* Exact arithmetic; floats corrupt money.
- *Why indexes on loans(member_id, copy_id)?* Foreign keys are not auto-indexed; we query and join on them constantly.

## 15. Proof: how this design kills the 3 bugs

Close the loop for your class. Revisit each anomaly from section 3:

| Bug | Before (messy table) | After (normalised + constrained) |
|---|---|---|
| **Update:** Ada moves | Address in N loan rows; miss one, data contradicts itself | Address in ONE `members` row. One UPDATE. All loans point at it. Impossible to half-update. |
| **Insert:** new book, no loans yet | Must invent a fake loan row | INSERT into `books` alone. No loan needed. No fake data. |
| **Delete:** Chinedu's loan ends | Deleting the loan row deletes his phone number | DELETE (or rather, set `return_date` on) the `loans` row. His `members` row and phone survive untouched. |

Plus the NEW protections constraints add:

- Cannot borrow a non-existent book (foreign key rejects it).
- Cannot enter a negative fine (CHECK rejects it).
- Cannot register two members with one email (UNIQUE rejects it).
- Cannot file a loan without a due date (NOT NULL rejects it).

> **The punchline for your talk: normalisation removes the TEMPTATION to write inconsistent data; constraints remove the ABILITY. You need both.**

## 16. Friday build walkthrough: foreign key + rejected insert (psql)

Your exact Friday task: *Add a foreign key and show a rejected bad insert.* Rehearse this flow so you can do it live and explain it before merge (team rule 1).

```bash
# 0. Start Postgres and connect (adapt to your setup)
psql -h localhost -U postgres -d library
```

```sql
-- 1. Minimal parents (if tables are fresh)
INSERT INTO members (full_name, email) VALUES ('Ada (fictional)', 'ada@example.com');
INSERT INTO books (title, author, isbn) VALUES ('Clean Code', 'Robert Martin', '9780132350884');
INSERT INTO copies (book_id, barcode) VALUES (1, 'COPY-0001');

-- 2. Create loans WITHOUT the FK first (to show the ADD step)
CREATE TABLE loans (
  loan_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  member_id BIGINT NOT NULL,
  copy_id BIGINT NOT NULL,
  loan_date DATE NOT NULL DEFAULT CURRENT_DATE,
  due_date DATE NOT NULL,
  return_date DATE
);

-- 3. ADD the foreign key (THE build step)
ALTER TABLE loans
  ADD CONSTRAINT fk_loans_member FOREIGN KEY (member_id) REFERENCES members(member_id)
  ON DELETE RESTRICT;
ALTER TABLE loans
  ADD CONSTRAINT fk_loans_copy FOREIGN KEY (copy_id) REFERENCES copies(copy_id)
  ON DELETE RESTRICT;

-- 4. Good insert: real member + real copy -> 1 row accepted
INSERT INTO loans (member_id, copy_id, due_date) VALUES (1, 1, CURRENT_DATE + 14);

-- 5. BAD insert: ghost member 9999 -> REJECTED (screenshot this)
INSERT INTO loans (member_id, copy_id, due_date) VALUES (9999, 1, CURRENT_DATE + 14);
-- ERROR: insert or update on table "loans" violates foreign key constraint "fk_loans_member"
-- DETAIL: Key (member_id)=(9999) is not present in table "members".

-- 6. Bonus: prove RESTRICT - try deleting a member with loans -> REJECTED
DELETE FROM members WHERE member_id = 1;
-- ERROR: update or delete on table "members" violates foreign key constraint ...

-- 7. Show your constraints to the reviewer
SELECT conname, contype FROM pg_constraint WHERE conrelid = 'loans'::regclass;
```

**What to say while demoing:** *The ALTER TABLE added a permanent rule. The good insert proves normal use still works. The bad insert proves the database - not my Python - enforces truth. The failed delete proves history is protected. My AI-use log and trade-off note are in the PR.*

**Failure experiment (team rule 4, pick this one):** run the bad insert BEFORE adding the FK and show it wrongly succeeding; then ADD the FK and re-run to show rejection. That contrast teaches why the constraint matters.

## 17. Beginner traps + honest trade-offs

Your team rules REQUIRE trade-offs in every answer (rule 6) and honest limits (rule 7). Memorise these:

### 5 traps beginners fall into

1. **Only validating in Python.** Forgetting the DB constraint. Then a script, a race condition or psql writes garbage. Fix: both layers.
2. **CASCADE everything.** One careless delete wipes history. Fix: default to RESTRICT for money/history; reserve CASCADE for truly owned children.
3. **Natural primary keys.** Using email/ISBN as PK, then it changes and chaos follows. Fix: surrogate ID + UNIQUE on the natural value.
4. **NULL vs empty string confusion.** `''` means *known to be empty*; NULL means *unknown*. For `return_date`, NULL correctly means *not returned yet*. Do not use magic dates like `1900-01-01`.
5. **Over-normalising.** Splitting `cities` when you only ever display a free-text city, or splitting tables nobody queries separately. Every split costs a JOIN. Normalise to kill REAL duplication bugs, not imaginary ones.

### 4 trade-offs to sound professional

| Choice | Gain | Cost |
|---|---|---|
| Normalise to 3NF | No duplication bugs, tiny updates | More JOINs; reads slightly slower/complex |
| Denormalise (e.g. store `book_title` on each loan row for speed) | Faster reads, fewer joins | Must keep copies in sync; risk of contradiction. Only do it with MEASURED slowness + a sync plan. |
| Strict constraints (FK/CHECK/UNIQUE) | Data always trustworthy | Writes can fail; app must handle DB errors gracefully |
| Surrogate keys | Stable, narrow joins | One extra column; natural lookups still need UNIQUE index |

> **One-line honesty for your README/release: normalised to 3NF; money as integer cents; history protected with RESTRICT; cross-row limits (max 5 active loans) are NOT enforced by CHECK and remain app-level - listed as a known limit.**

## 18. Cheat sheet (memorise this page)

```text
NORMALISATION: one fact, one home.
  1NF: one value per cell, every row has an ID.
  2NF: no column depends on PART of the key -> move it out.
  3NF: no non-key depends on a non-key -> move it out.
  Oath: the key, the whole key, nothing but the key.

ANOMALIES (U-I-D):
  Update: change in one copy, miss the other -> contradiction.
  Insert: cannot add X without fake Y.
  Delete: deleting X murders Y.

CONSTRAINTS:
  PRIMARY KEY = UNIQUE + NOT NULL + index. Surrogate IDs.
  FOREIGN KEY = must exist in parent. Default RESTRICT for history.
  UNIQUE = no two real values alike (NULLs exempt).
  NOT NULL = mandatory. NULL = unknown, not zero/empty.
  CHECK = one-row yes/no rule, NAMED. DEFAULT = auto-fill.

LIBRARY SHAPE:
  books 1--* copies 1--* loans *--1 members; loans 1--* fines
  money = INTEGER cents. FKs indexed. RESTRICT on history.
```

### Your next actions

1. Rehearse section 18's 15-minute script out loud once.
2. Run section 16's psql demo until the rejected insert feels boring.
3. Copy section 19 into your question-round notes; add 2 of your own.
4. Friday: PR with the FK migration + screenshot of the rejection + AI-use log + known limits.
5. Weekend SQL track: 1 hour on pgexercises.com (joins + aggregates) - it compounds.

You now know enough to teach this cold. The messy table, the 3 stories, the split, the strict librarian. That is the whole chunk.